## Check the Auxtel Spectrum PWV data and convert it into parquet

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys
import json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
from astropy.time import Time

In [ ]:
# ── publication-style rc params ───────────────────────────────────────────────
mpl.rcParams.update(
    {
        "figure.dpi": 150,
        "font.family": "serif",
        "font.size": 11,
        "axes.labelsize": 13,
        "axes.titlesize": 13,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 10,
        "axes.grid": True,
        "grid.alpha": 0.35,
        "axes.linewidth": 1.1,
    }
)

In [ ]:
# Enable interactive matplotlib backend if ipympl is available
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found -> interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found -> %matplotlib inline")

In [ ]:
def make_date_axis(ax_mjd, mjd_min, mjd_max, n_ticks=8):
    """Add a secondary top x-axis showing calendar dates (YYYY-MM)."""
    ax_date = ax_mjd.twiny()
    ax_date.set_xlim(ax_mjd.get_xlim())
    tick_mjds = np.linspace(mjd_min, mjd_max, n_ticks)
    tick_labels = [Time(m, format="mjd", scale="utc").strftime("%Y-%m") for m in tick_mjds]
    ax_date.set_xticks(tick_mjds)
    ax_date.set_xticklabels(tick_labels, rotation=30, ha="left", fontsize=9)
    ax_date.set_xlabel("Date (UTC)", fontsize=10, labelpad=6)
    return ax_date

In [ ]:
PATH_DIRDATA = "data_auxtelspectro"
PATH_FILEDATA_IN = "keep_auxtel_atmosphere_feb26_gaiaspec_gaiatarget_calspecthroughput_filteredtightcuts.csv"
PATH_FILEDATA_OUT = (
    "keep_auxtel_atmosphere_feb26_gaiaspec_gaiatarget_calspecthroughput_filteredtightcuts.parquet"
)
PATH_FULLFILEDATA_IN = os.path.join(PATH_DIRDATA, PATH_FILEDATA_IN)
PATH_FULLFILEDATA_OUT = os.path.join(PATH_DIRDATA, PATH_FILEDATA_OUT)

In [ ]:
df = pd.read_csv(PATH_FULLFILEDATA_IN)

In [ ]:
band_color = {"empty": "b", "OG550_65mm_1": "r"}

In [ ]:
mjd_min = df["MJD"].min()
mjd_max = df["MJD"].max()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
for band_sel in ["empty", "OG550_65mm_1"]:
    df_sub = df[df["FILTER"] == band_sel]
    band_color
    ax.errorbar(
        df_sub["MJD"].values,
        df_sub["PWV [mm]_rum"].values,
        yerr=df_sub["PWV [mm]_err_rum"].values,
        fmt="o",
        ms=2.5,
        color=band_color[band_sel],
        ecolor=band_color[band_sel],
        alpha=0.75,
        elinewidth=0.7,
        capsize=1.5,
        label=band_sel,
    )
ax.set_ylim(0.0, 20)
make_date_axis(ax, mjd_min, mjd_max, n_ticks=8)

In [ ]:
df.to_parquet(PATH_FULLFILEDATA_OUT)